In [1]:
# Install dependencies
!pip install "rfdetr[train,loggers]" supervision -qq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.1/588.1 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 69.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 77.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 143.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 89.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 159.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# Import and unzip the dataset
from google.colab import drive
import torch


# Mount Google Drive
drive.mount('/content/drive')

# Define the path to the zip file in Google Drive
dataset_name = "UFDD v3"#"WIDER Face Det 704 Enhanced v1"
zip_file_path = f'/content/drive/MyDrive/Instawork - Phase 1/{dataset_name}.zip'

# Unzip file
dst_path = f"/content/"
!unzip -q "{zip_file_path}" -d "{dst_path}"

Mounted at /content/drive


In [ ]:
import torch
from rfdetr import RFDETRLarge

device = (
    'cuda' if torch.cuda.is_available() else
    'mps' if torch.backends.mps.is_available() else
    'cpu'
)
print('Using device:', device)

max_faces = 40
model = RFDETRLarge(device=device, num_select=max_faces, num_queries=max_faces, num_classes=2)

Using device: cuda
[2026-06-03 18:56:01] [INFO] rf-detr - Downloading pretrained weights for /root/.roboflow/models/rf-detr-large-2026.pth


/root/.roboflow/models/rf-detr-large-2026.pth:   0%|          | 0.00/130M [00:00<?, ?iB/s]

[2026-06-03 18:56:03] [INFO] rf-detr - MD5 validation successful for /root/.roboflow/models/rf-detr-large-2026.pth


[2026-06-03 18:56:03] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-06-03 18:56:03] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


[2026-06-03 18:56:04] [INFO] rf-detr - File /root/.roboflow/models/rf-detr-large-2026.pth already exists with correct MD5 hash.


[2026-06-03 18:56:04] [WARNING] rf-detr - Checkpoint has 90 classes but model is configured for 2. The detection head will be re-initialized to 2 classes.
[2026-06-03 18:56:04] [WARNING] rf-detr - load_pretrain_weights: checkpoint lacks args.num_queries / args.group_detr; falling back to flat slice. With group_detr=13 this may scramble per-group query structure if the checkpoint was trained with group_detr > 1.


In [ ]:
LR_BASE = 5e-4
# LR_ENCODER_BASE = 1.5e-4
# lr_scale_factor = 1
output_dir = "./output_rf_detr"

aug_config = {
    "VerticalFlip": {
        "p": 0.1,
    },
    "HorizontalFlip": {
        "p": 0.40,
    },
    "RandomBrightnessContrast": {
        "brightness_limit": 0.2,
        "contrast_limit": 0.1,
        "p": 0.30,
    },
    "HueSaturationValue": {
        "hue_shift_limit": 8,
        "sat_shift_limit": 15,
        "val_shift_limit": 10,
        "p": 0.20,
    },
    "Affine": {
        "scale": [0.9, 1.1],
        "translate_percent": [-0.05, 0.05],
        "rotate": [-8, 8],
        "shear": [-3, 3],
        "p": 0.20,
    },
}

model.train(
    dataset_dir=dataset_name,
    epochs=15,
    batch_size=16,
    grad_accum_steps=1,
    lr=LR_BASE,
    # lr_encoder=LR_ENCODER_BASE * lr_scale_factor,
    early_stopping=True,
    early_stopping_min_delta=0.0001,
    early_stopping_patience=5,
    output_dir=output_dir,
    num_workers=4,
    checkpoint_interval=1,
    multi_scale=False,
    num_queries=max_faces,
    num_select=max_faces,
    aug_config=aug_config,
    # resolution=800,
)

[2026-06-03 18:56:05] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-06-03 18:56:05] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


[2026-06-03 18:56:06] [INFO] rf-detr - File /root/.roboflow/models/rf-detr-large-2026.pth already exists with correct MD5 hash.


[2026-06-03 18:56:06] [WARNING] rf-detr - Checkpoint has 90 classes but model is configured for 2. The detection head will be re-initialized to 2 classes.
[2026-06-03 18:56:06] [WARNING] rf-detr - load_pretrain_weights: checkpoint lacks args.num_queries / args.group_detr; falling back to flat slice. With group_detr=13 this may scramble per-group query structure if the checkpoint was trained with group_detr > 1.
INFO:pytorch_lightning.utilities.rank_zero:Using bfloat16 Automatic Mixed Precision (AMP)
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


[2026-06-03 18:56:10] [INFO] rf-detr - Building Roboflow train dataset with square resize at resolution 704
[2026-06-03 18:56:10] [INFO] rf-detr - Built 1 Albumentations transforms from config
[2026-06-03 18:56:10] [INFO] rf-detr - Built 5 Albumentations transforms from config
loading annotations into memory...
Done (t=0.02s)
creating index...
index created!
[2026-06-03 18:56:10] [INFO] rf-detr - Building Roboflow val dataset with square resize at resolution 704
[2026-06-03 18:56:10] [INFO] rf-detr - Built 1 Albumentations transforms from config
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!


/usr/local/lib/python3.12/dist-packages/pytorch_lightning/callbacks/model_checkpoint.py:881: Checkpoint directory /content/output_rf_detr exists and is not empty.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.utilities.rank_zero:Loading `train_dataloader` to estimate number of stepping batches.
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/model_summary/model_summary.py:242: Precision bf16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


...

[2026-06-03 19:01:53] [INFO] rf-detr - Best regular mAP saved to /content/output_rf_detr/checkpoint_best_regular.pth (epoch 5)
[2026-06-03 19:01:54] [INFO] rf-detr - Best EMA mAP improved to 0.5988 (epoch 5)


INFO:pytorch_lightning.callbacks.early_stopping:Metric __rfdetr_effective_map__ improved by 0.006 >= min_delta = 0.0001. New best score: 0.605


[2026-06-03 19:02:50] [INFO] rf-detr - Best EMA mAP improved to 0.6049 (epoch 6)


INFO:pytorch_lightning.callbacks.early_stopping:Monitored metric __rfdetr_effective_map__ did not improve in the last 5 records. Best score: 0.605. Signaling Trainer to stop.


[2026-06-03 19:07:32] [INFO] rf-detr - Best total checkpoint saved from EMA (regular=0.5873, ema=0.6049)


In [ ]:
import shutil
from pathlib import Path
from datetime import datetime

# Define source paths from the recent training run
train_output_dir = Path(output_dir)
ema_model_path = train_output_dir / "checkpoint_best_ema.pth"
metrics_path = train_output_dir / "metrics.csv"
training_config_path = train_output_dir / "training_config.json"

# Define the destination directory in Google Drive
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
drive_dest_dir = Path(f"/content/drive/MyDrive/Instawork - Phase 1/RF-DETR runs/{timestamp}")
drive_dest_dir.mkdir(parents=True, exist_ok=True)

# Copy the EMA model
if ema_model_path.exists():
    model_dst_path = drive_dest_dir / "checkpoint_best_ema.pth"
    shutil.copy(ema_model_path, model_dst_path)
    print(f"Successfully saved EMA model to {model_dst_path}")
else:
    print(f"Warning: {ema_model_path} not found.")

# Copy the metrics.csv
if metrics_path.exists():
    shutil.copy(metrics_path, drive_dest_dir / "metrics.csv")
    print(f"Successfully saved metrics.csv to {drive_dest_dir}")
else:
    print(f"Warning: {metrics_path} not found.")

# Copy the training config
if training_config_path.exists():
    shutil.copy(training_config_path, drive_dest_dir / "training_config.json")
    print(f"Successfully saved training config to {drive_dest_dir}")
else:
    print(f"Warning: {training_config_path} not found.")

# Copy TensorBoard events files
events_files = list(train_output_dir.glob("events.out.tfevents.*"))
for event_file in events_files:
    shutil.copy(event_file, drive_dest_dir / event_file.name)
    print(f"Successfully saved {event_file.name} to {drive_dest_dir}")
if not events_files:
    print(f"Warning: No events.out.tfevents.* files found in {train_output_dir}.")


Successfully saved EMA model to /content/drive/MyDrive/Instawork - Phase 1/RF-DETR runs/2026-06-03_19-07-32/checkpoint_best_ema.pth
Successfully saved metrics.csv to /content/drive/MyDrive/Instawork - Phase 1/RF-DETR runs/2026-06-03_19-07-32
Successfully saved training config to /content/drive/MyDrive/Instawork - Phase 1/RF-DETR runs/2026-06-03_19-07-32
Successfully saved events.out.tfevents.1780512970.ff606c27a368.820.0 to /content/drive/MyDrive/Instawork - Phase 1/RF-DETR runs/2026-06-03_19-07-32


## Inference on videos folders

In [20]:
import cv2
import supervision as sv
from pathlib import Path

from rfdetr import RFDETRLarge
from rfdetr_split_predict import RFDETRSplitPredictWrapper

WEIGHTS_PATH = f"/content/drive/MyDrive/Instawork - Phase 1/RF-DETR runs/2026-06-03_19-07-32/checkpoint_best_ema.pth"

def load_model() -> RFDETRLarge:
    if WEIGHTS_PATH:
        weights_path = Path(WEIGHTS_PATH).expanduser()
        if not weights_path.exists():
            raise FileNotFoundError(f"Weights file not found: {weights_path}")
        return RFDETRLarge(
            pretrain_weights=str(weights_path),
            num_classes=2,
            num_queries=40,
            num_select=40,
        )
    return RFDETRLarge()


model = RFDETRSplitPredictWrapper(
    load_model(),
    batch_size=32,  # must match CROP batching below (16 frames x 2 tiles)
)
box_annotator = sv.BoxAnnotator(thickness=1)
label_annotator = sv.LabelAnnotator(text_scale=0.35, text_thickness=1, text_padding=4)

[2026-06-03 20:42:15] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-06-03 20:42:15] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-06-03 20:42:16] [WARNING] rf-detr - load_pretrain_weights: args.num_queries absent; inferred ckpt_num_queries=40 from tensor rows 520 ÷ ckpt_group_detr=13.


In [21]:
def annotate_frames(frames_bgr, threshold):
    if not frames_bgr:
        return [], []

    rgb_frames = [
        cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB) for frame_bgr in frames_bgr
    ]

    detections_list = model.predict(rgb_frames, threshold=threshold)
    if not isinstance(detections_list, list):
        detections_list = [detections_list]

    annotated_frames = []
    for frame_bgr, detections in zip(frames_bgr, detections_list):
        annotated = box_annotator.annotate(scene=frame_bgr.copy(), detections=detections)

        for i in range(len(detections)):
            single_det = detections[i : i + 1]
            x1, y1, x2, y2 = single_det.xyxy[0]
            conf = float(single_det.confidence[0])
            label = f"{conf:.2f}"

            position = (
                sv.Position.BOTTOM_CENTER if y1 < 30 else sv.Position.TOP_CENTER
            )
            temp_annotator = sv.LabelAnnotator(
                text_scale=0.35,
                text_thickness=1,
                text_padding=4,
                text_position=position,
            )
            annotated = temp_annotator.annotate(
                scene=annotated, detections=single_det, labels=[label]
            )

        annotated_frames.append(annotated)

    return annotated_frames, detections_list

In [22]:
import time
import shutil
import zipfile
from pathlib import Path

ZIP_PATH = Path("/content/drive/MyDrive/Instawork - Phase 1/face-trimmed-videos.zip")
INPUT_DIR = Path("/content/input_videos")
OUTPUT_DIR = Path("/content/inference_output")

INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

THRESHOLD = 0.25
# 16 full-res frames -> up to 32 tiles per wrapper batch (batch_size=32 on wrapper)
BATCH_SIZE = 16

if not ZIP_PATH.exists():
    raise FileNotFoundError(f"Zip file not found: {ZIP_PATH}")

print("Unzipping videos...")
with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(INPUT_DIR)

video_paths = list(INPUT_DIR.rglob("*.mp4"))
print(f"\nFound {len(video_paths)} videos to process.")

for video_path in video_paths:
    print(f"\nProcessing: {video_path.name}")
    video_capture = cv2.VideoCapture(str(video_path))
    if not video_capture.isOpened():
        print(f"Failed to open video: {video_path}")
        continue

    fps = video_capture.get(cv2.CAP_PROP_FPS) or 30.0
    width = int(video_capture.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(video_capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_count = int(video_capture.get(cv2.CAP_PROP_FRAME_COUNT))

    video_output_path = OUTPUT_DIR / f"{video_path.stem}_rfdetr.mp4"
    video_writer = cv2.VideoWriter(
        str(video_output_path),
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (width, height),
    )

    if not video_writer.isOpened():
        video_capture.release()
        print(f"Failed to create output video: {video_output_path}")
        continue

    processed_frames = 0
    start_time = time.perf_counter()
    batch_frames = []

    while True:
        success, frame_bgr = video_capture.read()
        if not success:
            if batch_frames:
                annotated_frames, _ = annotate_frames(batch_frames, THRESHOLD)
                for f in annotated_frames:
                    video_writer.write(f)
                processed_frames += len(batch_frames)
            break

        batch_frames.append(frame_bgr)

        if len(batch_frames) == BATCH_SIZE:
            annotated_frames, _ = annotate_frames(batch_frames, THRESHOLD)
            for f in annotated_frames:
                video_writer.write(f)
            processed_frames += len(batch_frames)
            batch_frames = []

            total = frame_count if frame_count > 0 else "?"
            print(f"  Processed {processed_frames}/{total} frames")

    video_capture.release()
    video_writer.release()

    elapsed_seconds = time.perf_counter() - start_time
    effective_fps = processed_frames / elapsed_seconds if elapsed_seconds > 0 else 0.0
    print(f"Finished {video_path.name} - Effective FPS: {effective_fps:.2f}")

print("\nZipping inference results...")
shutil.make_archive("/content/batch_inference_results", "zip", OUTPUT_DIR)
print("All done! Results saved to /content/batch_inference_results.zip")

Unzipping videos...

Found 24 videos to process.

Processing: 11AFB0FE-BBFC-4169-8CFA-117BD49555F2_75AB9CEF-C484-47CF-A233-71555E673DEA_clip01_3p00s-11p00s.mp4
  Processed 16/240 frames
  Processed 32/240 frames
  Processed 48/240 frames
  Processed 64/240 frames
  Processed 80/240 frames
  Processed 96/240 frames
  Processed 112/240 frames
  Processed 128/240 frames
  Processed 144/240 frames
  Processed 160/240 frames
  Processed 176/240 frames
  Processed 192/240 frames
  Processed 208/240 frames
  Processed 224/240 frames
  Processed 240/240 frames
Finished 11AFB0FE-BBFC-4169-8CFA-117BD49555F2_75AB9CEF-C484-47CF-A233-71555E673DEA_clip01_3p00s-11p00s.mp4 - Effective FPS: 21.13

Processing: 14ADAD19-DBB1-4387-8FEA-5AB276568F86_7E0F16FB-C4E4-46A1-8A36-8D6739F1FE06_clip04_321p00s-325p00s.mp4
  Processed 16/121 frames
  Processed 32/121 frames
  Processed 48/121 frames
  Processed 64/121 frames
  Processed 80/121 frames
  Processed 96/121 frames
  Processed 112/121 frames
Finished 14ADA

## False Positives and False Negatives Analysis
Find images in the training dataset that have more than 4 False Positives (FP) or more than 4 False Negatives (FN).

In [ ]:
import csv
import json
from pathlib import Path

import cv2
import numpy as np
import supervision as sv
import torch
from rfdetr import RFDETRLarge
from torchvision.ops import box_iou

# Configuration
RUN_FP = True  # False to skip false positive analysis
RUN_FN = True  # False to skip false negative analysis
FN_CONFIDENCE_THRESHOLD = 0.25  # FN analysis: unmatched GTs vs preds at this threshold
FP_CONFIDENCE_THRESHOLD = 0.5  # FP analysis: unmatched preds at this threshold
MIN_ERRORS_TO_SAVE = 1
IOU_THRESHOLD = 0.1
BATCH_SIZE = 32
MAX_DETECTIONS = 40
DATASET_NAME = dataset_name
TRAIN_DIR = Path(f"/content/{DATASET_NAME}/train")
COCO_ANN_FILE = TRAIN_DIR / "_annotations.coco.json"
PRETRAIN_WEIGHTS = "/content/drive/MyDrive/Instawork - Phase 1/RF-DETR runs/2026-06-02_16-56-22/checkpoint_best_ema.pth"

OUTPUT_BASE = Path("/content/hard_examples")
FP_DIR = OUTPUT_BASE / "False_Positives"
FN_DIR = OUTPUT_BASE / "False_Negatives"

FP_CSV = OUTPUT_BASE / "fp_counts.csv"
FN_CSV = OUTPUT_BASE / "fn_counts.csv"
BOTH_CSV = OUTPUT_BASE / "fp_and_fn_counts.csv"
FP_BOXES_JSONL = (
    OUTPUT_BASE / "fp_boxes.jsonl"
)  # missing faces to merge into COCO later

if RUN_FP:
    FP_DIR.mkdir(parents=True, exist_ok=True)
if RUN_FN:
    FN_DIR.mkdir(parents=True, exist_ok=True)

if RUN_FP and RUN_FN:
    PREDICT_THRESHOLD = min(FN_CONFIDENCE_THRESHOLD, FP_CONFIDENCE_THRESHOLD)
elif RUN_FP:
    PREDICT_THRESHOLD = FP_CONFIDENCE_THRESHOLD
else:
    PREDICT_THRESHOLD = FN_CONFIDENCE_THRESHOLD

# Load model
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
model = RFDETRLarge(
    pretrain_weights=PRETRAIN_WEIGHTS,
    device=device,
    num_classes=2,
    num_queries=MAX_DETECTIONS,
    num_select=MAX_DETECTIONS,
)

# Load COCO Annotations
print(f"Loading annotations from {COCO_ANN_FILE}...")
with open(COCO_ANN_FILE, "r") as f:
    coco_data = json.load(f)

# Map image id to file name and ground truth boxes
images_info = {img["id"]: img["file_name"] for img in coco_data["images"]}
gt_boxes = {img["id"]: [] for img in coco_data["images"]}

for ann in coco_data["annotations"]:
    img_id = ann["image_id"]
    x, y, w, h = ann["bbox"]
    gt_boxes[img_id].append([x, y, x + w, y + h])  # Convert xywh to xyxy

# Annotators
box_annotator_fp = sv.BoxAnnotator(color=sv.Color.RED, thickness=2)
box_annotator_fn = sv.BoxAnnotator(color=sv.Color.BLUE, thickness=2)
label_annotator = sv.LabelAnnotator(text_scale=0.5, text_padding=4)

modes = []
if RUN_FP:
    modes.append("FP")
if RUN_FN:
    modes.append("FN")
print(
    f"Running {' + '.join(modes)} analysis on {len(images_info)} "
    f"training images in batches of {BATCH_SIZE}..."
)

processed = 0
batch_rgb = []
batch_meta = []
image_fp_counts = {}
image_fn_counts = {}


def greedy_match(pred_boxes, gts):
    """Greedy IoU matching. Returns (fp_indices, fn_indices) into pred_boxes and gts."""
    pred_boxes = np.asarray(pred_boxes)
    gts = list(gts) if len(gts) else []

    if len(pred_boxes) == 0 and len(gts) > 0:
        return [], list(range(len(gts)))
    if len(gts) == 0 and len(pred_boxes) > 0:
        return list(range(len(pred_boxes))), []
    if len(pred_boxes) == 0 or len(gts) == 0:
        return [], []

    pred_tensor = torch.tensor(pred_boxes, dtype=torch.float32)
    gt_tensor = torch.tensor(gts, dtype=torch.float32)
    iou_matrix = box_iou(pred_tensor, gt_tensor)

    matched_preds = set()
    matched_gts = set()
    ious = iou_matrix.flatten()
    sorted_indices = torch.argsort(ious, descending=True)

    for idx in sorted_indices:
        val = ious[idx]
        if val < IOU_THRESHOLD:
            break
        p_idx = (idx // iou_matrix.shape[1]).item()
        g_idx = (idx % iou_matrix.shape[1]).item()
        if p_idx not in matched_preds and g_idx not in matched_gts:
            matched_preds.add(p_idx)
            matched_gts.add(g_idx)

    fp_indices = [i for i in range(len(pred_boxes)) if i not in matched_preds]
    fn_indices = [i for i in range(len(gts)) if i not in matched_gts]
    return fp_indices, fn_indices


def xyxy_to_xywh(box):
    x1, y1, x2, y2 = box
    return [float(x1), float(y1), float(x2 - x1), float(y2 - y1)]


def write_fp_box_records(
    fp_boxes_file, file_name, fp_pred_boxes, fp_indices, fp_confidences
):
    for i in fp_indices:
        record = {
            "file_name": file_name,
            "bbox": xyxy_to_xywh(fp_pred_boxes[i]),
            "confidence": round(float(fp_confidences[i]), 4)
            if fp_confidences is not None
            else None,
            "source": "fp_analysis",
        }
        fp_boxes_file.write(json.dumps(record) + "\n")


def write_counts_csv(path, rows, fieldnames):
    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def save_count_csvs(fp_counts, fn_counts):
    if RUN_FP:
        fp_rows = [
            {"image": name, "fp_count": count} for name, count in fp_counts.items()
        ]
        fp_rows.sort(key=lambda r: r["fp_count"], reverse=True)
        write_counts_csv(FP_CSV, fp_rows, ["image", "fp_count"])

    if RUN_FN:
        fn_rows = [
            {"image": name, "fn_count": count} for name, count in fn_counts.items()
        ]
        fn_rows.sort(key=lambda r: r["fn_count"], reverse=True)
        write_counts_csv(FN_CSV, fn_rows, ["image", "fn_count"])

    if RUN_FP and RUN_FN:
        both_names = {
            name
            for name in fp_counts
            if fp_counts[name] >= MIN_ERRORS_TO_SAVE
            and fn_counts.get(name, 0) >= MIN_ERRORS_TO_SAVE
        }
        both_rows = [
            {
                "image": name,
                "fp_count": fp_counts[name],
                "fn_count": fn_counts[name],
            }
            for name in both_names
        ]
        both_rows.sort(
            key=lambda r: (r["fp_count"], r["fn_count"]),
            reverse=True,
        )
        write_counts_csv(BOTH_CSV, both_rows, ["image", "fp_count", "fn_count"])


def process_batch(batch_rgb, batch_meta, processed_count, fp_boxes_file=None):
    if not batch_rgb:
        return processed_count

    preds_list = model.predict(batch_rgb, threshold=PREDICT_THRESHOLD)

    if not isinstance(preds_list, list):
        preds_list = [preds_list]

    for meta, preds in zip(batch_meta, preds_list):
        img_id, file_name, image_bgr, gts = meta
        pred_boxes = preds.xyxy
        confidences = preds.confidence

        if RUN_FN:
            _, fn_indices = greedy_match(pred_boxes, gts)
            fn_count = len(fn_indices)
            image_fn_counts[file_name] = fn_count
            if fn_count >= MIN_ERRORS_TO_SAVE:
                fn_detections = sv.Detections(
                    xyxy=gts[fn_indices], class_id=np.zeros(len(fn_indices), dtype=int)
                )
                labels = ["FN (Missed)"] * len(fn_indices)
                annotated_fn = box_annotator_fn.annotate(
                    scene=image_bgr.copy(), detections=fn_detections
                )
                annotated_fn = label_annotator.annotate(
                    scene=annotated_fn, detections=fn_detections, labels=labels
                )
                cv2.imwrite(str(FN_DIR / file_name), annotated_fn)

        if RUN_FP:
            if confidences is not None:
                fp_mask = confidences >= FP_CONFIDENCE_THRESHOLD
                fp_pred_boxes = pred_boxes[fp_mask]
                fp_confidences = confidences[fp_mask]
            else:
                fp_pred_boxes = pred_boxes
                fp_confidences = None

            fp_indices, _ = greedy_match(fp_pred_boxes, gts)
            fp_count = len(fp_indices)
            image_fp_counts[file_name] = fp_count
            if fp_boxes_file is not None and fp_count > 0:
                write_fp_box_records(
                    fp_boxes_file,
                    file_name,
                    fp_pred_boxes,
                    fp_indices,
                    fp_confidences,
                )
            if fp_count >= MIN_ERRORS_TO_SAVE:
                fp_detections = sv.Detections(
                    xyxy=fp_pred_boxes[fp_indices],
                    confidence=fp_confidences[fp_indices]
                    if fp_confidences is not None
                    else None,
                    class_id=np.zeros(len(fp_indices), dtype=int),
                )
                labels = (
                    [f"FP {conf:.2f}" for conf in fp_detections.confidence]
                    if fp_detections.confidence is not None
                    else ["FP"] * len(fp_detections)
                )
                annotated_fp = box_annotator_fp.annotate(
                    scene=image_bgr.copy(), detections=fp_detections
                )
                annotated_fp = label_annotator.annotate(
                    scene=annotated_fp, detections=fp_detections, labels=labels
                )
                cv2.imwrite(str(FP_DIR / file_name), annotated_fp)

        processed_count += 1
        if processed_count % 100 == 0:
            print(f"  Processed {processed_count}/{len(images_info)} images...")

    return processed_count


fp_boxes_file = None
if RUN_FP:
    OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
    fp_boxes_file = open(FP_BOXES_JSONL, "w", encoding="utf-8")

for img_id, file_name in images_info.items():
    img_path = TRAIN_DIR / file_name
    if not img_path.exists():
        continue

    image_bgr = cv2.imread(str(img_path))
    if image_bgr is None:
        continue

    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    gts = np.array(gt_boxes.get(img_id, []))

    batch_rgb.append(image_rgb)
    batch_meta.append((img_id, file_name, image_bgr, gts))

    if len(batch_rgb) == BATCH_SIZE:
        processed = process_batch(batch_rgb, batch_meta, processed, fp_boxes_file)
        batch_rgb = []
        batch_meta = []

if batch_rgb:
    processed = process_batch(batch_rgb, batch_meta, processed, fp_boxes_file)

if fp_boxes_file is not None:
    fp_boxes_file.close()

save_count_csvs(image_fp_counts, image_fn_counts)

print(f"\nAnalysis complete! Processed {processed} images.")
if RUN_FP:
    print(f"False Positives saved in: {FP_DIR}")
    print(f"FP counts CSV: {FP_CSV}")
    print(f"Missing face boxes (JSONL): {FP_BOXES_JSONL}")
if RUN_FN:
    print(f"False Negatives saved in: {FN_DIR}")
    print(f"FN counts CSV: {FN_CSV}")
if RUN_FP and RUN_FN:
    print(f"FP+FN overlap CSV: {BOTH_CSV}")


[2026-06-02 17:27:32] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-06-02 17:27:32] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-06-02 17:27:33] [WARNING] rf-detr - load_pretrain_weights: args.num_queries absent; inferred ckpt_num_queries=40 from tensor rows 520 ÷ ckpt_group_detr=13.
[2026-06-02 17:27:33] [WARNING] rf-detr - Model is not optimized for inference. Latency may be higher than expected. You can optimize the model for inference by calling model.optimize_for_inference().


Loading annotations from /content/UFDD/train/_annotations.coco.json...
Running FP + FN analysis on 1500 training images in batches of 32...


  Processed 100/1500 images...
  Processed 200/1500 images...
  Processed 300/1500 images...
  Processed 400/1500 images...
  Processed 500/1500 images...
  Processed 600/1500 images...
  Processed 700/1500 images...
  Processed 800/1500 images...
  Processed 900/1500 images...
  Processed 1000/1500 images...
  Processed 1100/1500 images...
  Processed 1200/1500 images...
  Processed 1300/1500 images...
  Processed 1400/1500 images...
  Processed 1500/1500 images...

Analysis complete! Processed 1500 images.
False Positives saved in: /content/hard_examples/False_Positives
FP counts CSV: /content/hard_examples/fp_counts.csv
Missing face boxes (JSONL): /content/hard_examples/fp_boxes.jsonl
False Negatives saved in: /content/hard_examples/False_Negatives
FN counts CSV: /content/hard_examples/fn_counts.csv
FP+FN overlap CSV: /content/hard_examples/fp_and_fn_counts.csv


In [ ]:
import shutil
from pathlib import Path

# Define the directory to zip and the output zip file name
folder_to_zip = "/content/hard_examples"
output_zip_path = "/content/hard_examples"

# Create the zip file
print(f"Zipping {folder_to_zip}...")
shutil.make_archive(output_zip_path, 'zip', folder_to_zip)
print(f"Successfully created zip file at {output_zip_path}.zip")

!cp /content/hard_examples.zip "/content/drive/MyDrive/Instawork - Phase 1/."

Zipping /content/hard_examples...
Successfully created zip file at /content/hard_examples.zip
